# Laboratorio 7 - Visión Por Computadora

Autores:

- Nelson García
- Joaquín Puente
- Diego Linares



# Task 1

## Pregunta 1.1


El médico coordinador del proyecto le presenta la siguiente situación: "Tenemos 800 radiografías
etiquetadas por nuestros radiólogos. Un colega me dijo que con tan pocos datos el modelo va a memorizar
todo y no va a servir para nada en producción."

Como ingeniero de IA a cargo, usted decide aplicar Data Augmentation como parte de la solución. Sin
embargo, el médico le pregunta: "¿No estamos inventando datos falsos que podrían confundir al modelo?"
Con esto en mente, responda las siguientes preguntas en su reporte:

1. Explíquele al médico, en términos que él pueda entender, qué es el Data Augmentation y por qué las
imágenes generadas no son datos falsos. Use la analogía que considere más apropiada.

R//

El Data Augmentation es una técnica para ampliar artificialmente el conjunto de entrenamiento aplicando transformaciones controladas a imágenes reales ya etiquetadas. La idea no es inventar pacientes ni patologías nuevas, sino presentar la misma evidencia clínica en variaciones plausibles que podrían ocurrir durante la adquisición de la imagen, como pequeños cambios de posición, escala o contraste. En tus diapositivas se describe justamente que el Data Augmentation aplica transformaciones T(x) a imágenes existentes para expandir el espacio de entrada y evitar que la red memorice píxeles específicos del dataset, mejorando así la generalización.

Una analogía útil para explicárselo al médico es la siguiente: es como si un residente aprendiera a reconocer una neumonía viendo la misma radiografía en diferentes condiciones de visualización: un poco más clara, un poco más oscura, ligeramente desplazada o con una pequeña variación de encuadre. La enfermedad no cambia; lo que cambia es la forma en que se presenta la imagen. Por eso, las imágenes aumentadas no son datos falsos, sino variantes válidas de datos reales, siempre que las transformaciones respeten la anatomía y no alteren el significado diagnóstico. El objetivo de este tipo de regularización es que el modelo aprenda patrones robustos y no dependa de detalles accidentales del set de entrenamiento.

En otras palabras, el modelo no debería aprender “esta radiografía exacta”, sino conceptos más generales como distribución de opacidades, tamaño cardíaco, patrón intersticial o consolidaciones. Esa es precisamente la lógica detrás de usar augmentación y otras técnicas de regularización para reducir sobreajuste.

2. En el contexto específico de radiografías de tórax, proponga tres transformaciones de Data
Augmentation que serían válidas y justifique cada una. Luego, identifique una transformación que
no debería aplicarse en este dominio médico y explique por qué podría comprometer la integridad
diagnóstica del modelo.
R//

Primero, pequeñas traslaciones o recortes leves.
Son válidos porque en la práctica clínica el encuadre nunca es idéntico: el paciente puede estar levemente centrado a la izquierda o derecha, o el campo radiográfico puede variar un poco entre estudios. Entrenar con estas variaciones ayuda a que el modelo no dependa de una posición exacta del pulmón dentro de la imagen y aprenda a reconocer hallazgos aunque cambie ligeramente la ubicación global. Esto es consistente con la idea de que el augmentation debe introducir invariancia útil sin alterar el contenido clínico.

Segundo, rotaciones pequeñas.
Una radiografía real puede presentar una inclinación leve por postura del paciente o por adquisición imperfecta. Permitir rotaciones pequeñas hace al modelo más tolerante a ese tipo de variación operativa. La clave es que sean pequeñas; no se trata de girar la placa de manera extrema, sino de simular desalineaciones realistas de adquisición.

Tercero, ajustes moderados de brillo o contraste.
En imagen médica puede haber diferencias de exposición, procesamiento digital o visualización entre equipos y centros. Una modificación moderada del contraste o intensidad puede ayudar a que el modelo reconozca el mismo patrón patológico bajo distintas condiciones de adquisición o posprocesamiento. Debe hacerse con prudencia, porque el propósito es simular variabilidad real del sistema, no deformar la apariencia de los hallazgos. Tus materiales del curso mencionan explícitamente modificaciones de brillo y contraste como formas comunes de augmentation.

Una transformación que no debería aplicarse en este dominio es el flip horizontal.
La razón es que en tórax la lateralidad importa clínicamente: invertir izquierda y derecha puede cambiar la interpretación anatómica, la localización de dispositivos, la posición cardíaca, la distribución de derrames o la referencia de lesiones unilaterales. Aunque la imagen siga “pareciendo” un tórax, su significado clínico ya no sería fiel al caso original. En este contexto sí se estaría comprometiendo la integridad diagnóstica del dato, porque la transformación altera información semántica importante para el diagnóstico. Por eso, no toda transformación geométrica válida en visión por computadora general es válida en imagen médica.

3. ¿Es el Data Augmentation suficiente por sí solo para garantizar que el modelo generalice bien?
Argumente su posición considerando otras variables del proceso de entrenamiento.

R//

No. El Data Augmentation ayuda mucho, pero no es suficiente por sí solo para garantizar que el modelo generalice bien. La generalización depende también de la calidad del dataset, del balance entre clases, de la separación correcta entre entrenamiento/validación/prueba, de la arquitectura elegida, de la regularización, del optimizador, del learning rate y, especialmente en este caso, del uso de transfer learning por tratarse de un conjunto pequeño de solo 800 estudios. Tus diapositivas del curso lo dejan claro: con pocos datos etiquetados, la estrategia robusta no es usar augmentation aislado, sino combinarlo con transferencia de aprendizaje, fine-tuning, Adam y técnicas de regularización como L2, dropout y early stopping.

Además, el sobreajuste no aparece solo por “tener pocos datos”, sino también porque el modelo puede aprender correlaciones espurias o detalles irrelevantes del conjunto de entrenamiento. La literatura de visión y deep learning insiste en que regularización y validación son necesarias para evitar que el modelo ajuste ruido o peculiaridades del dataset en vez de aprender patrones verdaderos.

En este proyecto, el argumento más sólido sería decir que el Data Augmentation es una pieza de la solución, no la solución completa. Con 800 radiografías, una estrategia razonable sería partir de un modelo preentrenado, congelar o reutilizar capas tempranas que ya capturan bordes, texturas y estructuras visuales generales, y luego hacer fine-tuning cuidadoso sobre el dominio médico. Ese es justamente el valor del transfer learning cuando el conjunto etiquetado es limitado: reduce requerimientos de datos, acelera el entrenamiento y suele mejorar la capacidad de generalización frente a entrenar desde cero.

Por tanto, mi posición es la siguiente: el Data Augmentation es necesario y útil, pero no garantiza por sí solo una buena generalización. Para que el modelo sea confiable en producción también se necesita un buen diseño experimental, validación externa si es posible, control del sobreajuste, regularización adecuada y transferencia de aprendizaje adaptada al dominio de radiografías.

## Pregunta 1.2

Durante el entrenamiento del modelo de MediScan Guatemala, el equipo registra las siguientes curvas de
pérdida (loss) al finalizar 25 épocas:

Con base a estos datos, responda dentro de su reporte:

4. Identifique a partir de qué época aproximada comienza el sobreajuste (overfitting) y describa cómo
se evidencia en los números de la tabla.

5. Proponga dos estrategias de regularización concretas (por ejemplo, Dropout o L2) que podría haber
implementado para prevenir este comportamiento. Para cada una, explique intuitivamente qué
fenómeno matemático está mitigando y qué impacto esperaría ver en las curvas de la tabla.

6. Desde una perspectiva médica, ¿por qué es especialmente peligroso desplegar en producción un
modelo que exhibe este patrón de sobreajuste para el diagnóstico de radiografías? Argumente más
allá de los números.

## Pregunta 1.3

Un inversionista que asiste a la demo del proyecto le pregunta: "¿Por qué su modelo tiene un 94% de
accuracy en las pruebas? ¿Eso significa que es confiable para diagnosticar neumonía?"
Usted sabe que el dataset de prueba contiene 700 radiografías normales y solo 150 radiografías con
neumonía.

Responda lo siguiente en su reporte:

7. Calcule cuál sería el accuracy de un modelo naive que simplemente predice siempre 'Normal' para
todas las imágenes. ¿Qué revela ese cálculo sobre el 94% reportado?

8. Explique por qué, en problemas médicos con clases desbalanceadas, métricas como el F1-Score o
la Sensibilidad (Recall) para la clase minoritaria son más informativas que el accuracy. No se limite
a definirlas; argumente su relevancia clínica.

9. Como director técnico, ¿cómo le respondería al inversionista de forma honesta y profesional?
Redacte una respuesta breve (3 a 5 oraciones) que sea técnicamente sólida pero comprensible para
un no-especialista.